In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
%sql
SELECT * FROM dbw_gitarchive.silver.github_events LIMIT 2

In [0]:
dirty_df = spark.sql("select * FROM dbw_gitarchive.silver.github_events")
dirty_df.show(10)

In [0]:
#Whether there are duplicates or not, we will still clean it up for the future refrence
#First, remove the records that have nulls in id, as that is not useful for us

no_null_df = dirty_df.filter(F.col("event_id").isNotNull())


now, we check for the duplicates


In [0]:
def remove_duplicates(df, primary_col, order_col):
    window_dup = Window.partitionBy(primary_col).orderBy(order_col)
    df = df.withColumn("rank", F.rank().over(window_dup))
    df = df.filter(F.col("rank") == 1)
    df = df.drop("rank")
    return df

In [0]:
silver_cleaned_df = remove_duplicates(no_null_df, "event_id", "processed_at")

In [0]:
silver_cleaned_df.show()